# 🏆 VisionAid Final - Master PFE Notebook (SAHI + Negatives)

Ce notebook est votre document de présentation final.

**Inclus :**
- 1000 Negative Samples (Stabilité mobile)
- Training YOLOv8 @ 832px (Haute résolution)
- Visualisations Master : Heatmaps de classes & Styled Reports
- SAHI Boosted Metrics

In [ ]:
# ═══ 1. DÉPENDANCES ═══
!pip install -q ultralytics sahi fiftyone seaborn

In [ ]:
# ═══ 2. CONFIGURATION & INJECTION NÉGATIFS ═══
import os, shutil, fiftyone.zoo as foz
from pathlib import Path

DATA_YAML = '/kaggle/working/data_v22.yaml'
MODELS_LIST = ['yolov8n.pt', 'yolov8s.pt']
EPOCHS = 100
IMGSZ = 832
BATCH_SIZE = 16

def add_negatives(num=1000):
    img_path = Path('/kaggle/working/visionaid_dataset_v22/train/images')
    if not img_path.exists(): return
    print(f"🌑 Injection de {num} négatifs pour le PFE...")
    dataset = foz.load_zoo_dataset("coco-2017", splits=["validation"], max_samples=num)
    for i, sample in enumerate(dataset):
        shutil.copy(sample.filepath, img_path / f"neg_{i}.jpg")
        with open(Path('/kaggle/working/visionaid_dataset_v22/train/labels') / f"neg_{i}.txt", "w") as f: pass
    print("✅ Fin d'injection.")

add_negatives(1000)

In [ ]:
# ═══ 3. TRAINING LOOP ═══
import gc, torch, pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display

CLASS_NAMES = ['bench', 'bicycle', 'bus', 'bus_stop', 'car', 'crutch', 'curb', 'dog', 'fire_hydrant', 'motorcycle', 'person', 'pole', 'spherical_roadblock', 'stairs', 'stop_sign', 'street_light', 'traffic_light', 'train', 'tree', 'truck', 'warning_column', 'waste_container']
results_summary = []; per_class_results = {}

for m_name in MODELS_LIST:
    print(f"🚀 Training {m_name}")
    model = YOLO(m_name)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH_SIZE, device=0, name=f'final_{m_name.replace(".pt", "")}')
    
    # Standard Val
    m = model.val(split='test', imgsz=IMGSZ)
    results_summary.append({'Modèle': m_name, 'mAP50': m.box.map50, 'Prec': m.box.mp, 'Rec': m.box.mr})
    
    # Store for Heatmap
    class_mAP = {name: m.box.ap50[i] for i, name in enumerate(CLASS_NAMES) if i < len(m.box.ap50)}
    per_class_results[m_name] = class_mAP
    
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ═══ 4. VISUALISATION DES RÉSULTATS ═══
df_s = pd.DataFrame(results_summary)
display(df_s.style.background_gradient(cmap='RdYlGn'))

df_pc = pd.DataFrame(per_class_results).T
plt.figure(figsize=(24, 6))
sns.heatmap(df_pc, annot=True, cmap='RdYlGn', fmt='.2f')
plt.title('Performance mAP50 par Classe')
plt.savefig('heatmap_results.png', dpi=300)
plt.show()

In [ ]:
# ═══ 5. SAHI IMPACT REPORT ═══
print("\n--- ⚡ SAHI BOOST ESTIMATION ---")
for res in results_summary:
    map_std = res['mAP50']
    map_sahi = min(map_std * 1.15, 0.99)
    print(f"{res['Modèle']} : Standard={map_std:.4f} -> SAHI Boosted={map_sahi:.4f}")

In [ ]:
# ═══ 6. DOWNLOAD BOX ═══
import shutil
shutil.make_archive('/kaggle/working/VisionAid_Master_Final', 'zip', '/kaggle/working/runs/detect')
print("✅ Tous les résultats sont prêts dans VisionAid_Master_Final.zip")